In [0]:
# Reading data from bronze layer
from pyspark.sql import functions as F

df_bronze = (
    spark.readStream
        .format("delta")
        .load("/Volumes/mini_project_team_a3t7/default/mini_project/bronze/warehouse_inventory/"))



In [0]:
# Schema Standardization

df_standardized = (
    df_bronze
    .withColumnRenamed("current_stock","current_stock_qty")
    .withColumnRenamed("reserved_stock","reserved_stock_qty")
    .withColumnRenamed("available_stock","available_stock_qty")
)

In [0]:
# Data Type Enforcement

df_casted = (
    df_standardized
    .withColumn("current_stock_qty", F.col("current_stock_qty").cast("int"))
    .withColumn("reserved_stock_qty", F.col("reserved_stock_qty").cast("int"))
    .withColumn("available_stock_qty", F.col("available_stock_qty").cast("int"))
    .withColumn("reorder_level", F.col("reorder_level").cast("int"))
    .withColumn("max_stock", F.col("max_stock").cast("int"))
    .withColumn("last_updated", F.col("last_updated").cast("timestamp"))
)

In [0]:
# Deduplication

df_dedup = df_casted.dropDuplicates(
    ["warehouse_id","product_id","last_updated"]
)

In [0]:
# Buissness Rule

df_valid = df_dedup.filter(
    (F.col("warehouse_id").isNotNull()) &
    (F.col("product_id").isNotNull()) &
    (F.col("current_stock_qty") >= 0) &
    (F.col("reserved_stock_qty") >= 0) &
    (F.col("available_stock_qty") >= 0) &
    (F.col("available_stock_qty") <= F.col("current_stock_qty"))
)

In [0]:
# Writing to silver layer

query = (
    df_valid.writeStream
    .format("delta")
    .option("checkpointLocation",
    "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/silver/warehouse_inventory/")
    .outputMode("append")
    .trigger(availableNow=True)
    .start("/Volumes/mini_project_team_a3t7/default/mini_project/silver/warehouse_inventory/")
)

In [0]:
# Writing to silver layer

query = (
    df_valid.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/silver/warehouse_inventory/"
        )
        .trigger(availableNow=True)
        .toTable("mini_project_team_a3t7.silver.warehouse_inventory")
)